In [24]:
!pwd

/kaggle/working/ARC-AGI-3_dev


In [25]:
!nvidia-smi

Wed Sep  9 13:45:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
%pip install -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of cuda-python to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 71.1 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 85.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━

In [17]:
!ps aux | grep -E "vllm|api_server" | grep -v grep

root         225 10.6  6.1 8780100 2018280 ?     Sl   12:25   0:48 /usr/bin/python3 -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-VL-2B-Instruct --served-model-name arc-local --host 127.0.0.1 --port 1234 --dtype half --max-model-len 8192 --gpu-memory-utilization 0.90


In [18]:
import requests

r = requests.get(
    "http://127.0.0.1:1234/v1/models",
    timeout=10,
)

print(r.status_code)
print(r.json())

200
{'object': 'list', 'data': [{'id': 'arc-local', 'object': 'model', 'created': 1788957192, 'owned_by': 'vllm', 'root': 'Qwen/Qwen3-VL-2B-Instruct', 'parent': None, 'max_model_len': 8192, 'permission': [{'id': 'modelperm-a938da1ad348222f', 'object': 'model_permission', 'created': 1788957192, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


In [19]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:1234/v1",
    api_key="local",
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": "Reply exactly with: LOCAL MODEL WORKS",
        }
    ],
    temperature=0,
    max_tokens=50,
)

print(response.choices[0].message.content)

LOCAL MODEL WORKS


In [21]:
from PIL import Image, ImageDraw
import io
import base64

image = Image.new(
    "RGB",
    (256, 256),
    "white",
)

draw = ImageDraw.Draw(image)

draw.rectangle(
    (50, 50, 200, 200),
    fill="red",
)

buffer = io.BytesIO()
image.save(buffer, format="PNG")

encoded = base64.b64encode(
    buffer.getvalue()
).decode("ascii")

image_url = (
    "data:image/png;base64,"
    + encoded
)

response = client.chat.completions.create(
    model="arc-local",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What shape and color do you see?",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                    },
                },
            ],
        }
    ],
    temperature=0,
    max_tokens=100,
)

print(response.choices[0].message.content)

Based on the image provided, I can see the following:

-   **Shape:** The object is a square.
-   **Color:** The color is red.

The image is a simple, solid red square.


In [31]:
import os

os.environ["ARC_LOCAL_BASE_URL"] = (
    "http://127.0.0.1:1234/v1"
)

os.environ["ARC_LOCAL_MODEL"] = (
    "arc-local"
)

In [2]:
%cd /kaggle/working
!git clone https://github.com/14yamahi/ARC-AGI-3_dev.git

/kaggle/working
Cloning into 'ARC-AGI-3_dev'...
remote: Enumerating objects: 734, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 734 (delta 0), reused 0 (delta 0), pack-reused 728 (from 2)
Receiving objects: 100% (734/734), 489.41 KiB | 13.59 MiB/s, done.
Resolving deltas: 100% (490/490), done.


In [5]:
%cd /kaggle/working/ARC-AGI-3_dev

!ls

/kaggle/working/ARC-AGI-3_dev
agents	 llms.txt  pyproject.toml  README.md  uv.lock
LICENSE  main.py   pytest.ini	   tests


In [6]:
import os

os.environ["ARC_LOCAL_BASE_URL"] = (
    "http://127.0.0.1:1234/v1"
)
os.environ["ARC_LOCAL_MODEL"] = "arc-local"

In [18]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ["ARC_API_KEY"] = secrets.get_secret("ARC_API_KEY")

In [21]:
!git pull
!MPLBACKEND=Agg UV_LINK_MODE=copy uv run main.py --agent=myagent2 --game=ls20

Already up to date.
https://arcprize.org/api/games
2026-09-09 13:42:36,249 | INFO | Game list: ['ls20-9607627b']
2026-09-09 13:42:36 | INFO | Successfully fetched 25 environment(s) from API
2026-09-09 13:42:36,419 | INFO | Successfully fetched 25 environment(s) from API
***** MAKING SCORECARD
{"card_id":"001448db-219d-4bd3-bc9f-267d08ea5b2e"}

{'card_id': '001448db-219d-4bd3-bc9f-267d08ea5b2e'}
2026-09-09 13:42:36 | INFO | Created new scorecard: 001448db-219d-4bd3-bc9f-267d08ea5b2e
2026-09-09 13:42:36,592 | INFO | Created new scorecard: 001448db-219d-4bd3-bc9f-267d08ea5b2e
***** MAKING ALL AGENTS with card id: 001448db-219d-4bd3-bc9f-267d08ea5b2e
2026-09-09 13:42:37 | INFO | Successfully fetched metadata for game ls20
2026-09-09 13:42:37,024 | INFO | Successfully fetched metadata for game ls20
2026-09-09 13:42:37 | INFO | Successfully reset game ls20-9607627b, guid=be95ebe2-8ff0-40ce-8a2a-1ca14d66dab7, scorecard_id=001448db-219d-4bd3-bc9f-267d08ea5b2e
2026-09-09 13:42:37,278 | INFO | S

In [22]:
!ls /kaggle/working/ARC-AGI-3_dev/debug_frames

debug_scene.png
